In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import yaml


# 26_580_643 rows
with open("configs.yaml", "r") as f:
    configs = yaml.safe_load(f)

In [2]:

iid = 0
SAVE_OUT_PATH = configs["PATH_TO_DATA_DIR"]
ALL_COLLECTIONS = configs["ALL_COLLECTIONS"]
path_to_data = f"{SAVE_OUT_PATH}/nuGun_pT_0_50_reco_{iid}.h5"
df_head = pd.read_hdf(path_to_data, key="df", stop=10)
print(df_head)


   event                    collection      Edep          t           x  \
0      0  InnerTrackerBarrelCollection  0.000028   4.001957 -503.471520   
1      0  InnerTrackerBarrelCollection  0.000035   3.993593 -503.892225   
2      0  InnerTrackerBarrelCollection  0.000125   2.825821 -549.297765   
3      0  InnerTrackerBarrelCollection  0.000079   2.749115  159.376294   
4      0  InnerTrackerBarrelCollection  0.000204   2.748186  159.384792   
5      0  InnerTrackerBarrelCollection  0.000036   2.741781  160.211095   
6      0  InnerTrackerBarrelCollection  0.000101   0.799673   61.521157   
7      0  InnerTrackerBarrelCollection  0.000059  14.911034 -127.009552   
8      0  InnerTrackerBarrelCollection  0.000062   2.214822  243.244621   
9      0  InnerTrackerBarrelCollection  0.000026   0.857233  116.482556   

            y           z  system  side  layer  module  sensor    cellid0  \
0 -231.515346 -617.124690       3     0      2      70       2   34128131   
1 -230.534940 -615.5

In [4]:
# Count unique cell IDs and hits per cell ID for inside_bounds hits, per subcollection
from collections import defaultdict

cell_hit_counts = {col: defaultdict(int) for col in ALL_COLLECTIONS}

for chunk in pd.read_hdf(path_to_data, key="df", chunksize=1_000_000):
    for col in ALL_COLLECTIONS:
        mask = (chunk["collection"] == col) & (chunk["inside_bounds"] == True)
        sub = chunk.loc[mask, "cellid0"]
        for cell_id, count in sub.value_counts().items():
            cell_hit_counts[col][cell_id] += count

for col in ALL_COLLECTIONS:
    counts = cell_hit_counts[col]
    if counts:
        n_unique = len(counts)
        hits_per_cell = list(counts.values())
        min_hits = min(hits_per_cell)
        min_cells = [cid for cid, c in counts.items() if c == min_hits]
        print(f"{col}: {n_unique} unique cell IDs, hits per cell — min={min_hits}, max={max(hits_per_cell)}, mean={sum(hits_per_cell)/n_unique:.2f}")
        print(f"  cell IDs with min hits ({min_hits}): {min_cells}")
    else:
        print(f"{col}: 0 unique cell IDs")

print(cell_hit_counts["OuterTrackerBarrelCollection"][470155525])

OuterTrackerBarrelCollection: 64512 unique cell IDs, hits per cell — min=3, max=180, mean=50.85
  cell IDs with min hits (3): [86540549, 34971909, 196869]
OuterTrackerEndcapCollection: 1152 unique cell IDs, hits per cell — min=849, max=1791, mean=1272.42
  cell IDs with min hits (849): [150995430, 67109158, 402653606]
InnerTrackerBarrelCollection: 9032 unique cell IDs, hits per cell — min=100, max=3339, mean=434.70
  cell IDs with min hits (100): [958723]
InnerTrackerEndcapCollection: 1196 unique cell IDs, hits per cell — min=73, max=3906, mean=1350.07
  cell IDs with min hits (73): [318767972]
VertexBarrelCollection: 485 unique cell IDs, hits per cell — min=1299, max=12883, mean=4390.03
  cell IDs with min hits (1299): [33719041]
VertexEndcapCollection: 256 unique cell IDs, hits per cell — min=8782, max=23942, mean=16001.45
  cell IDs with min hits (8782): [218104738]
23


In [ ]:

OUTPUT_COLS = ["Edep", "x", "y", "z", "t", "system", "side", "layer", "module", "sensor"]

results = {col: [] for col in ALL_COLLECTIONS}
counts_all = {col: 0 for col in ALL_COLLECTIONS}

for chunk in pd.read_hdf(path_to_data, key="df", chunksize=1_000_000):
    for col in ALL_COLLECTIONS:
        mask_all = chunk["collection"] == col
        counts_all[col] += mask_all.sum()
        mask = mask_all & (chunk["inside_bounds"] == True)
        results[col].append(chunk.loc[mask, OUTPUT_COLS])

for col in ALL_COLLECTIONS:
    print(f"length of {col}, all: {counts_all[col]}")
    arr = pd.concat(results[col]).to_numpy() if results[col] else np.empty((0, len(OUTPUT_COLS)))
    print(f"length of {col}, inside_bounds: {len(arr)}")
    np.save(f"{SAVE_OUT_PATH}/{col}_SimTrackerHit_conditional_reco_{iid}.npy", arr)

In [1]:
import sys
print(sys.executable)

/software/python-anaconda-2022.05-el8-x86_64/bin/python
